> 在线讲义：[第 18 章 · 跑得快：异步推理、少看几眼、压小权重](https://wikieden.github.io/vla-from-scratch/lessons/18-fast-inference.html) · 本书《VLA 从零到 π0》

# 第 18 章 · 跑得快：异步推理、少看几眼、压小权重

配套讲义：仓库里 `vla-deep-dive/notes/lessons/18-fast-inference.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `min_threshold(latency, dt, n)` | 0.18 |
| 第 1 步 | `simulate_queue(g, latency, dt, n, seconds)` | g = 0 时每段停 0.3 s；g = 0.18 时不停 |
| 第 1 步 | `SMOLVLA_FIG5` | 快 29.5%；9 → 19 个 |
| 第 2 步 | `prefix_cost(tokens_per_frame, cams, layers)` | 13,824 vs 1,728，少 8 倍 |
| 第 3 步 | `weight_gb(params, bits)`、`OPENVLA_QUANT` | 14 / 7 / 3.5 GB |

数据取自 smolvla-2506.01844.txt 图 5、openvla-2406.09246.txt §5.4 表 2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
SMOLVLA_FIG5 = {"同步": {"time_s": 13.75, "cubes_60s": 9, "sr": 78.3}, "异步": {"time_s": 9.70, "cubes_60s": 19, "sr": 73.3}}
OPENVLA_QUANT = {"bf16": (16.8, 71.3), "int8": (10.2, 58.1), "int4": (7.0, 71.9)}  # 显存 GB、成功率 %

**函数 `min_threshold`**：不断粮的最小 g：往返期间要走 latency/dt 步，占一段 n 步的比例。

In [ ]:
def min_threshold(latency, dt, n):
    """不断粮的最小 g：往返期间要走 latency/dt 步，占一段 n 步的比例。"""
    return (latency / dt) / n

**函数 `simulate_queue`**：按控制步模拟动作队列；返回 (停着的秒数, 前向次数)。

In [ ]:
def simulate_queue(g, latency, dt, n, seconds):
    """按控制步模拟动作队列；返回 (停着的秒数, 前向次数)。

    每一步：① 新一段到了就换上（跳过发出观测后已经走掉的步数）；② 队列剩余比例 ≤ g 或为空时发观测；
    ③ 队列有动作就执行一个，没有就停一步。
    """
    lag = math.ceil(round(latency / dt, 9))
    queue, arrive, executed_since_send = n, None, 0
    idle = calls = 0
    for step in range(round(seconds / dt)):
        if arrive is not None and step >= arrive:
            queue, arrive = n - executed_since_send, None
        if arrive is None and (queue == 0 or queue / n <= g):
            arrive, executed_since_send, calls = step + lag, 0, calls + 1
        if queue > 0:
            queue -= 1
            executed_since_send += 1
        else:
            idle += 1
    return idle * dt, calls

**函数 `prefix_cost`**：前缀计算量的粗估：token 数 × 层数。

In [ ]:
def prefix_cost(tokens_per_frame, cams, layers):
    """前缀计算量的粗估：token 数 × 层数。"""
    return tokens_per_frame * cams * layers

**函数 `weight_gb`**：只算权重的显存（1 GB = 1e9 字节）。

In [ ]:
def weight_gb(params, bits):
    """只算权重的显存（1 GB = 1e9 字节）。"""
    return params * bits / 8 / 1e9

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 异步推理：阈值 g 与动作队列")
g_min = min_threshold(0.3, 1 / 30, 50)
print(f"  Δt = 1/30 s、n = 50、往返 0.3 s：最小 g = {g_min:.2f}")
for g in (0.0, g_min, 0.7, 1.0):
    idle, calls = simulate_queue(g, 0.3, 1 / 30, 50, 10)
    print(f"  g = {g:.2f}：10 秒内停了 {idle:.2f} s，前向 {calls} 次")
s, a = SMOLVLA_FIG5["同步"], SMOLVLA_FIG5["异步"]
print(f"  SmolVLA：{s['time_s']} s → {a['time_s']} s（快 {1 - a['time_s'] / s['time_s']:.1%}），"
      f"60 秒 {s['cubes_60s']} → {a['cubes_60s']} 个")

In [ ]:
print("\n第二步 前缀计算量 ∝ token × 层")
big, small = prefix_cost(256, 3, 18), prefix_cost(64, 3, 9)
print(f"  π0 式 {big:,}，SmolVLA 式 {small:,}，少 {big / small:.0f} 倍")

In [ ]:
print("\n第三步 量化显存（7B 只算权重）")
for bits in (16, 8, 4):
    print(f"  {bits:>2} 位：{weight_gb(7e9, bits):.1f} GB")
for k, (gb, sr) in OPENVLA_QUANT.items():
    print(f"  OpenVLA 实测 {k}：{gb} GB、成功率 {sr}%")

## 3. 代码里才看得到的

1. `simulate_queue` 按控制步离散地走，触发条件写成 ≤（论文写 <）：g 正好等于下限时，新一段在队列刚好用完的那一步到达，不断粮；用 < 会差一步。代码同一时间只允许一次推理在路上，所以 g = 1 也只是每 9 步发一次。
2. 新一段到达时，代码丢掉旧队列、从新一段里跳过"想的这段时间已经走掉的步数"，这是最简单的合并方式；SmolVLA 在重叠处做聚合。
3. `weight_gb` 用 1e9 字节当 1 GB，和 OpenVLA 表里的写法一致；按 GiB 算会小 7%。

## 4. 自己改一改

- 在 `simulate_queue` 里把往返时间改成随机的（比如 0.2–0.4 s 均匀），看 g = 0.18 还够不够。
- 用 `prefix_cost` 算"只减 token、不减层"和"只减层、不减 token"各省多少。
- 给 `weight_gb` 加上视觉编码器（0.4B、bf16），比较总显存。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 菜还剩三成就下新单，往返多久就留够几步；
> 读得长、读得遍数多就慢，少看几眼少算几层；
> 压成电子版省地方，翻页太慢反而误事。